In [ ]:
import os
from tqdm.notebook import tqdm


DATA_ROOT_DIR = '../data_m/compress_dataset'

REFERENCE_DIR = os.path.join(DATA_ROOT_DIR, 'references')

TRAIN_DIR = os.path.join(DATA_ROOT_DIR, 'train')
VALID_DIR = os.path.join(DATA_ROOT_DIR, 'valid')
TEST_DIR = os.path.join(DATA_ROOT_DIR, 'test')

references_names = [
    'canFam6.fa', # 狗
    # 'hg38.fa', # 人
    'mm39.fa', # 小鼠
    'Rnor6_0.fna', # 大鼠
    'equCab3.fa', # 马
    'indicum.fna', # 芝麻
]

samples_names = [
    # 'SRR6737547_1.fastq',
    # 'SRR6737547_2.fastq',
    # 'ERR3929511_1.fastq',
    # 'ERR3929511_2.fastq',
    # 'SRR12922210_1.fastq',
    # 'SRR12922210_2.fastq',
    # 'SRR12072893_1.fastq',
    # 'SRR12072893_2.fastq',
    # 'ERR3528872_1.fastq',
    # 'ERR3528872_2.fastq',
    # 'SRR12845693.fastq',
    # 'SRR15829874.fastq',
    # 'SRR14773546_1.fastq',
    # 'SRR14773546_2.fastq',
    # 'SRR3206414.fastq',
    # 'SRR12448025.fastq',
    # 'SRR5943529_subreads.fastq',
    # 'SRR11816799_1.fastq',
    # 'SRR11073097_1.fastq',
    # 'ERR2585114.fastq',
    # 'ERR1397639.fastq',
]

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)
os.makedirs(VALID_DIR, exist_ok=True)

In [8]:
def get_file_size(file_path):
    return os.path.getsize(file_path)

def all_files_size(file_names, dir):
    all_size = 0
    for file_name in file_names:
        file_path = os.path.join(dir, file_name)
        all_size += get_file_size(file_path)
    return all_size

def folder_files_size(folder):
    all_size = 0
    for root, dirs, files in os.walk(folder):
        for file in files:
            file_path = os.path.join(root, file)
            all_size += get_file_size(file_path)
    return all_size

In [9]:
from Bio import SeqIO
import random


# 从fastq文件中抽取碱基
def extract_base_from_fastq(fastq_file):
    with open(fastq_file, "r") as handle:
        try:
            for record in SeqIO.parse(handle, "fastq"):
                yield record.seq.upper()
        except ValueError as e:
            print(f"Skipping record due to error: {e}")

# 从fasta文件中抽取碱基
def extract_base_from_fasta(fasta_file):
    with open(fasta_file, "r") as handle:
        try:
            for record in SeqIO.parse(handle, "fasta"):
                yield record.seq.upper()
        except ValueError as e:
            print(f"Skipping record due to error: {e}")

reads_cnt_cache = {}
def get_reads_cnt(file):
    if file in reads_cnt_cache:
        return reads_cnt_cache[file]

    reads_cnt = 0
    if file.endswith('.fastq') or file.endswith('.fq'):
        with open(file, "r") as handle:
            try:
                for record in SeqIO.parse(handle, 'fastq'):
                    reads_cnt += 1
            except ValueError as e:
                print(f"Skipping record due to error: {e}")
    elif file.endswith('.fasta') or file.endswith('.fa') or file.endswith('.fna'):
        with open(file, "r") as handle:
            try:
                for record in SeqIO.parse(handle, 'fasta'):
                    reads_cnt += 1
            except ValueError as e:
                print(f"Skipping record due to error: {e}")
    reads_cnt_cache[file] = reads_cnt
    return reads_cnt


only_base_suffix = '.ob'


In [10]:

# 处理references
for reference_name in tqdm(references_names, desc='references'):
    reference_file = os.path.join(REFERENCE_DIR, reference_name)
    output_file = os.path.join(TRAIN_DIR, reference_name + only_base_suffix)
    with open(output_file, 'w') as f:
        for base in extract_base_from_fasta(reference_file):
            f.write(str(base) + '\n')

# 处理samples
for sample_name in tqdm(samples_names, desc='train samples'):
    sample_file = os.path.join(DATA_ROOT_DIR, sample_name)
    sample_size = get_file_size(sample_file)
    output_file = os.path.join(TRAIN_DIR, sample_name + only_base_suffix)
    with open(output_file, 'w') as f:
        read_cnt = get_reads_cnt(sample_file)
        # 随机读取第n行后的m行数据, m是read_cnt的0.5%
        generate_cnt = int(read_cnt * 0.005)
        start = random.randint(0, read_cnt//2)
        end = random.randint(start, start + generate_cnt)
        counter = 0
        for base in extract_base_from_fastq(sample_file):
            counter += 1
            if counter >= start:
                f.write(str(base) + '\n')
            if counter >= end:
                break

print('train size:', folder_files_size(TRAIN_DIR))

references:   0%|          | 0/1 [00:00<?, ?it/s]

train samples:   0%|          | 0/11 [00:00<?, ?it/s]

Skipping record due to error: Lengths of sequence and quality values differs for SRR12922210.40674666 40674666/2 (150 and 81).
Skipping record due to error: Lengths of sequence and quality values differs for SRR15829874.71259952 71259952/2 (100 and 146).
Skipping record due to error: Lengths of sequence and quality values differs for ERR2585114.718566 915d6bee-4237-4c72-ab89-a375fcadcdd7/1 (19716 and 19836).
train size: 6180512201


In [11]:
# 处理samples, 生成test
for sample_name in tqdm(samples_names, desc='test samples'):
    sample_file = os.path.join(DATA_ROOT_DIR, sample_name)
    sample_size = get_file_size(sample_file)
    output_file = os.path.join(TEST_DIR, sample_name + only_base_suffix)
    with open(output_file, 'w') as f:
        read_cnt = get_reads_cnt(sample_file)
        # 随机读取第n行后的m行数据, m是read_cnt的0.1%
        generate_cnt = int(read_cnt * 0.001)
        start = random.randint(0, read_cnt//2)
        end = random.randint(start, start + generate_cnt)
        counter = 0
        for base in extract_base_from_fastq(sample_file):
            counter += 1
            if counter >= start:
                f.write(str(base) + '\n')
            if counter >= end:
                break

print('test size:', folder_files_size(TEST_DIR))

test samples:   0%|          | 0/11 [00:00<?, ?it/s]

test size: 53142975


In [12]:
# 处理samples, 生成valid
for sample_name in tqdm(samples_names, desc='valid samples'):
    sample_file = os.path.join(DATA_ROOT_DIR, sample_name)
    sample_size = get_file_size(sample_file)
    output_file = os.path.join(VALID_DIR, sample_name + only_base_suffix)
    with open(output_file, 'w') as f:
        read_cnt = get_reads_cnt(sample_file)
        # 随机读取第n行后的m行数据, m是read_cnt的0.05%
        generate_cnt = int(read_cnt * 0.0005)
        start = random.randint(0, read_cnt//2)
        end = random.randint(start, start + generate_cnt)
        counter = 0
        for base in extract_base_from_fastq(sample_file):
            counter += 1
            if counter >= start:
                f.write(str(base) + '\n')
            if counter >= end:
                break

print('valid size:', folder_files_size(VALID_DIR))

valid samples:   0%|          | 0/11 [00:00<?, ?it/s]

valid size: 14834479


In [13]:
finetune_path = os.path.join(DATA_ROOT_DIR, 'fine_tune_val')
os.makedirs(finetune_path, exist_ok=True)

compress_path = os.path.join(DATA_ROOT_DIR, 'compress_val')
os.makedirs(compress_path, exist_ok=True)

# 生成fine_tune_val
for sample_name in tqdm(samples_names, desc='finetune samples'):
    sample_file = os.path.join(DATA_ROOT_DIR, sample_name)
    sample_size = get_file_size(sample_file)
    output_file = os.path.join(finetune_path, sample_name + only_base_suffix)
    with open(output_file, 'w') as f:
        read_cnt = get_reads_cnt(sample_file)
        # 随机读取第n行后的m行数据, m是read_cnt的0.01%
        generate_cnt = int(read_cnt * 0.0001)
        start = random.randint(0, read_cnt//2)
        end = random.randint(start, start + generate_cnt)
        counter = 0
        for base in extract_base_from_fastq(sample_file):
            counter += 1
            if counter >= start:
                f.write(str(base) + '\n')
            if counter >= end:
                break

print('fine_tune_val size:', folder_files_size(finetune_path))


finetune samples:   0%|          | 0/11 [00:00<?, ?it/s]

fine_tune_val size: 3632850


compress samples:   0%|          | 0/11 [00:00<?, ?it/s]

compress_val size: 20765174


In [14]:

# 生成compress_val
for sample_name in tqdm(samples_names, desc='compress samples'):
    sample_file = os.path.join(DATA_ROOT_DIR, sample_name)
    sample_size = get_file_size(sample_file)
    output_file = os.path.join(compress_path, sample_name + only_base_suffix)
    with open(output_file, 'w') as f:
        read_cnt = get_reads_cnt(sample_file)
        # 随机读取第n行后的m行数据, m是read_cnt的0.005%
        generate_cnt = int(read_cnt * 0.00005)
        start = random.randint(0, read_cnt//2)
        end = random.randint(start, start + generate_cnt)
        counter = 0
        for base in extract_base_from_fastq(sample_file):
            counter += 1
            if counter >= start:
                f.write(str(base) + '\n')
            if counter >= end:
                break
    
print('compress_val size:', folder_files_size(compress_path))



compress samples:   0%|          | 0/11 [00:00<?, ?it/s]

compress_val size: 1562561
